In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import ListedColormap

# Load the dataset
df = pd.read_csv('dataset_mansion.csv', index_col=0)

df['Consumption'] = pd.to_numeric(
    df['Consumption'].astype('string').str.replace(',', '.', regex=False),
    errors='raise'
)
df['MTime'] = pd.to_datetime(df['MTime'], errors='raise', utc=True)
assert df['MTime'].notna().all(), 'Check missing timestamps first.'
assert not df['MTime'].duplicated().any(), 'Check duplicate timestamps first.'
df = df.set_index('MTime').sort_index()

expected = pd.date_range(df.index.min(), df.index.max(), freq='h')
assert df.index.equals(expected), 'This analysis expects a complete hourly timeline.'

# Preserve the original dataframe
df_features = df.copy()

local_time = df_features.index.tz_convert('Europe/Helsinki')

# Target timestamp's hour: 0–23
hour = local_time.hour

df_features['hour_sin'] = np.sin(2 * np.pi * hour / 24)
df_features['hour_cos'] = np.cos(2 * np.pi * hour / 24)

# Weekday = 0, weekend = 1
df_features['is_weekend'] = (
    local_time.dayofweek >= 5
).astype(int)

# Month: convert January–December to 0–11 before encoding
month = local_time.month - 1

df_features['month_sin'] = np.sin(2 * np.pi * month / 12)
df_features['month_cos'] = np.cos(2 * np.pi * month / 12)

# # Inspect the added columns
# new_columns = [
#     'hour_sin',
#     'hour_cos',
#     'is_weekend',
#     'month_sin',
#     'month_cos'
# ]

# print(df_features[new_columns].head())

# Initial feature-removal choices.
# Edit this list to keep or remove other columns.
columns_to_drop = [
    'Latitude',
    'Longitude',
    'Place',
    'PresentWeather(rank)',
    'HighestTemperature(degC)',
    'LowestTemperature(degC)',
    'MaximumWindSpeed(m/s)',
    'MinimumWindSpeed(m/s)'
]

df_features = df_features.drop(columns=columns_to_drop)

print('Remaining columns:')
print(df_features.columns.tolist())

Remaining columns:
['Consumption', 'AirTemperature(degC)', 'RelativeHumidity(%)', 'WindSpeed(m/s)', 'WindDirection(deg)', 'PrecipitationAmount(mm)', 'MaximumPrecipitationIntensity(mm/h)', 'AirPressure(hPa)', 'hour_sin', 'hour_cos', 'is_weekend', 'month_sin', 'month_cos']
